## LLM-enhanced Linux Kernel Profiling/Enhancement

### Background

We want to know the benign kernel-level behaviors (a set of benign kernel functions) of each syscall.

We propose hybrid program analysis --- it combines dynamic profiling (which captures actual executed paths but may miss unexecuted flows) with static call graph analysis (which reveals all potential calls but often over-approximates) [1].

In complex software (e.g. OS kernels), dynamic call graphs can be under-approximated – many legitimate call edges remain unseen because the inputs or execution did not trigger them. 
Pure static analysis, on the other hand, may introduce spurious edges that are infeasible at runtime (false positives). 

The goal is to bridge this gap by expanding the dynamic call graph with likely edges suggested by static analysis, while filtering out unrealistic ones. Large Language Models (LLMs) can serve as semantic reasoners in this context. 

An LLM can assess whether a candidate static call edge (a function $L$ calling a function $x$) is plausible given: 
- (1) the current observed call graph context, 
- (2) the call path leading to $L$ (i.e. how the program reached $L$), and 
- (3) the *semantics and signatures* of both $L$ and $x$. 

By understanding code semantics and intent (learned from vast code corpora), the LLM can infer if, for example, function $L$ (say, a resource initializer) would logically invoke function $x$ (perhaps a resource finalizer or related helper) in that context. 
This semantic filtering helps eliminate static edges that don't make sense, reducing false positives while recovering missed edges that a pure dynamic profile didn’t cover. 
Prior work has shown the promise of LLMs in similar roles – e.g. using code summaries to match indirect call targets [2] or to prune static analysis false positives by asking about expected function behavior [1]. 
The key insight is that LLMs can interpret high-level semantic cues (function names, documentation, code comments, etc.) to validate control-flow relationships that purely syntax-based analyses might miss [3].

> [1] Poster: Assisting Static Analysis with Large Language Models: A ChatGPT Experiment (S\&P)
> 
> [2] Semantic-Enhanced Indirect Call Analysis with Large Language Models (arxiv)
>
> [3] https://github.com/247arjun/ai-static-analysis-pipeline/blob/main/Elevating%20Code%20Security%20and%20Reliability%20via%20LLM-Augmented%20Static%20Analysis.md


In [27]:
%load_ext autoreload
%autoreload 2
import networkx as nx
import matplotlib.pyplot as plt
import os
import time
import re
import subprocess
import scipy
from kfunc_filter import should_filter_function

from ollama import chat
from collections import defaultdict, deque

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


First of all, we have to load the static kernel call graph.

In [28]:
kernel_cg_path = "./callgraph.graphml"
kernel_cg_acyc_path = "./pruned_callgraph_filtered.graphml"

k_cg = nx.read_graphml(kernel_cg_path)
k_cg_acyc = nx.read_graphml(kernel_cg_acyc_path)

# prune self-loops
k_cg.remove_edges_from(nx.selfloop_edges(k_cg))

In this CallGraph (CG), each node is a *function name*, and each edge (u, v) denotes a *call relation* u (caller) -> v (callee).

For a single node u1, it may has different call targets (let's say {v1, v2, ..., vn}).

- If function u1 has indirect calls (u1 as the callsite), then it will has `u1.has_indirect = True`.
- If function u1 has indirect call targets, for each edge {(u1, v1), (u1, v2), ...}, the indirect call edge will have property `indirect = True`

See the demonstrated `describe_node` below.

In [29]:
def describe_node(graph, node):
    if node not in graph:
        print(f"Node {node} does not exist in the graph.")
        return

    print(f"Node: {node}")
    print(f"Has indirect calls: {graph.nodes[node].get('has_indirect', False)}")

    print("Outgoing edges:")
    for _, target, edge_data in graph.out_edges(node, data=True):
        indirect = edge_data.get('indirect', False)
        print(f"  {node} -> {target:<20} (is indirect call: {indirect})")

In [30]:
describe_node(k_cg_acyc, "filp_close")

Node: filp_close
Has indirect calls: True
Outgoing edges:
  filp_close -> dnotify_flush        (is indirect call: False)
  filp_close -> fput                 (is indirect call: False)
  filp_close -> mqueue_flush_file    (is indirect call: True)
  filp_close -> nfs4_file_flush      (is indirect call: True)
  filp_close -> nfs_file_flush       (is indirect call: True)


Read this kernel function (`filp_close`) source code:
```c
int filp_close(struct file *filp, fl_owner_t id)
{
	int retval = 0;

	if (!file_count(filp)) {
		printk(KERN_ERR "VFS: Close: file count is 0\n");
		return 0;
	}

	if (filp->f_op->flush)
		retval = filp->f_op->flush(filp, id);

	if (likely(!(filp->f_mode & FMODE_PATH))) {
		dnotify_flush(filp, id);
		locks_remove_posix(filp, id);
	}
	fput(filp);
	return retval;
}
```

From the source code, you can see it has several direct calls:
```
filp_close -> printk
filp_close -> dnotify_flush
filp_close -> locks_remove_posix
filp_close -> fput
```

And it has indirect call: `filp_close -> filp->f_op->flush()`.

In [31]:
# This u1 ("filp_close") is a node with indirect calls
describe_node(k_cg, "filp_close")

Node: filp_close
Has indirect calls: True
Outgoing edges:
  filp_close -> _printk              (is indirect call: False)
  filp_close -> dnotify_flush        (is indirect call: False)
  filp_close -> fput                 (is indirect call: False)
  filp_close -> locks_remove_posix   (is indirect call: False)
  filp_close -> mqueue_flush_file    (is indirect call: True)
  filp_close -> nfs4_file_flush      (is indirect call: True)
  filp_close -> nfs_file_flush       (is indirect call: True)


In [32]:
# This u2 ("__x64_sys_close") is a node without indirect calls. 
# This is the entry function of a syscall `close`.
describe_node(k_cg, "__x64_sys_close")

Node: __x64_sys_close
Has indirect calls: False
Outgoing edges:
  __x64_sys_close -> close_fd             (is indirect call: False)


### 1. Ask LLM to generate per-function summary

LLMs have been proven to show good performance of understanding code semantics and summarizing the behavior of functions [1-3].
> [1] Large Language Models for Code Analysis: Do LLMs Really Do Their Job? (Usenix Security'24)
>
> [2] kernelGPT: Enhanced Kernel Fuzzing via Large Language Models (ASPLOS'25)
>
> [3] Exploiting Code Symmetries for Learning Program Semantics (arxiv)

In this work, we would use Qwen3-32B (https://huggingface.co/Qwen/Qwen3-32B) (or any other state-of-the-art code LLM).


For every individual function, we need to request LLM to summarize its semantics, include the function definitions and behaviors.

**Input**: function source code (e.g., the above `filp_close` source code).

**Output**: function prototype and code semantics summary.

Here is demonstrated prompts:

**Prompt0**:
I will input Linux kernel function source code. Please read the source code and summarize its behavior.
My input: function definition and source code.
Your output:
```
{
   "func_name": input function, 
   "signature": its argument types and return value types,
   "summary": its behavior (within X words),
}.
```
**Prompt1:** filp_close {source code}

**Answer1:** 
```
{
   "func_name": "filp_close",
   "signature": "args: {struct file *, fl_owner_t}, ret: {int}",
   "source code": {...},
   "summary": "Closes a file descriptor by performing flush operations, removing file notifications, releasing POSIX locks, and decrementing the file reference count. Returns the result of the flush operation if implemented."
}
```

In [33]:
# Here are all kernel functions in the call graph
# You only need to consider functions that are not filtered out
# (we filter functions if they are related to interrupt handlers/context switches)

kall_functions = [node for node in k_cg.nodes if not should_filter_function(node)]
print(f"Total kernel functions (after filtering): {len(kall_functions)}/{len(k_cg.nodes)}")

Total kernel functions (after filtering): 41879/45596


### 2. Ask LLM to enhance our dynamic profile

A dynamic profile contains all performs benign behaviors (of a syscall) during our certain workloads. It would absolutely under-approximate the potential behaviors of that syscall.

We will input the dynamic profile into LLM to let it enlarge possible benign behaviors, by analyzing the code semantics and contexts.

**Some useful graph generation and traversal-related functionalities:**
- Generate a dynamic call graph (subgraph) from profiles
- Traverse a graph from a node (see its out edges)

In [34]:
def gen_subgraph(static_graph, sys_entry_function=None, function_set=None, hops=2):
    """
    if sys_entry_function is set, and function_set is None: 
        Generate a subgraph starting from the sys_entry_function, including all reachable nodes.
    
    if sys_entry_function is None, and function_set is set:
        Generate a subgraph containing only the nodes in function_set.
    
    if both sys_entry_function and function_set are set:
        This is a "tricky" indicator.
        Generate a subgraph containing the nodes in function_set, and also bfs traverse the nodes in N `hops`
        from each node in function_set.
    """
    if not function_set:
        # If function_set is empty, use all reachable nodes from the sys_entry_function
        subgraph_nodes = nx.descendants(static_graph, sys_entry_function)
        subgraph_nodes.add(sys_entry_function)  # Include the entry function itself
    else:
        # Filter nodes to include only those in the function_set
        function_set = [f for f in function_set if not should_filter_function(f)]
        subgraph_nodes = set(function_set)

    if sys_entry_function is not None and function_set is not None:
        # Include nodes in function_set and traverse `hops` from each node
        if hops >= 0:
            for node in function_set:
                visited = set()
                queue = [(node, 0)]  # (current_node, current_hop)
                while queue:
                    current_node, current_hop = queue.pop(0)
                    if current_hop < hops and current_node not in visited:
                        visited.add(current_node)
                        neighbors = list(static_graph.successors(current_node))
                        subgraph_nodes.update(neighbors)
                        queue.extend([(neighbor, current_hop + 1) for neighbor in neighbors])
        
        else:   # hops = -1: consider all descendant nodes of function_set
            for node in function_set:
                _desc = nx.descendants(static_graph, node)
                subgraph_nodes.update(_desc)
                
        # Mark all nodes in the original function_set as "profiled"
        for node in function_set:
            if node in static_graph.nodes:
                static_graph.nodes[node]["profiled"] = True

    # Create the subgraph with the filtered nodes
    subgraph = static_graph.subgraph(subgraph_nodes).copy()
    return subgraph

In [35]:
def outedges_callgraph(graph, function_name, do_filter=True):
    """ 
    function_name is the source node (u), it will return all out-edges (u, v1), (u, v1),... of this node.
    """
    if function_name not in graph:
        print(f"Function {function_name} does not exist in the graph.")
        return []

    out_edges = []
    for _, target, edge_data in graph.out_edges(function_name, data=True):
        if do_filter and should_filter_function(target):
            continue
        out_edges.append((function_name, target, edge_data))
    
    return out_edges

In [36]:
def graph_to_markdown_tree(graph):
    """
    Convert a directed graph to a markdown tree representation
    """
    def dfs(node, indent=0, visited=set()):
        lines = []
        prefix = "  " * indent + "- " + node
        if graph.nodes[node].get("profiled", False):
            prefix += " <profiled>"
        lines.append(prefix)
        visited.add(node)
        for _, neighbor in graph.out_edges(node):
            if neighbor not in visited:
                lines.extend(dfs(neighbor, indent + 1, visited))
        return lines
    roots = [n for n in graph.nodes if graph.in_degree(n) == 0]
    all_lines = []
    for root in roots:
        all_lines.extend(dfs(root, indent=0, visited=set()))
    return "\n".join(all_lines)

In [39]:
def draw_subgraph(graph, graph_name):
    plt.figure(figsize=(12, 8))
    pos = nx.spring_layout(graph)  # Use the graph layout
    in_degree_zero_nodes = [node for node in graph if graph.in_degree(node) == 0]
    node_colors = ['red' if node in in_degree_zero_nodes else 'lightblue' for node in graph.nodes]
    nx.draw(graph, pos, with_labels=True, node_size=500, font_size=10, node_color=node_colors, edge_color='black')
    print(f"Subgraph number of nodes: {len(graph.nodes())}, number of edges: {len(graph.edges())}")
    plt.title(f"Subgraph: {graph_name}")
    plt.show()


### Now, we process our profile data and produce outputs.

**Source directory and file output layout**.

In [40]:
# Directory paths for syscall profiles and processes

src_directory = "syscall_profiles/"
proc_direcotry = "syscall_procs/"

linux_src_directory = "linux/"
func_src_file = "the_functions_all.txt"

# statistics derived from pure static CFG
proc_pure_static_file = "pure-static.txt"

# statistics derived from static CFG based-on profiling information
proc_profile_static_file = "profile-static.txt"

In [41]:
# Let's fix the directory path now
syscall = "close"
sys_id = 3

s_src_dir = os.path.join(src_directory, f"{sys_id}:{syscall}")
s_proc_dir = os.path.join(proc_direcotry, f"{sys_id}:{syscall}")

In [42]:
# we extract the profiled functions by reading the syscall profile file
def extract_profiled_functions(profile_file):
    profiled_functions = set()
    with open(profile_file, "r") as f:
        lines = f.readlines()
        for line in lines:
            func_name = line.strip()
            if func_name in kall_functions and not should_filter_function(func_name):
                profiled_functions.add(func_name)
    return profiled_functions

# dump statistics (include numbers of nodes/edges, and its markdown tree) of the subgraph
def dump_subgraph_statistics(subgraph, file_name):
    with open(str(file_name), "w") as f:
        f.write(f"Subgraph has {len(subgraph.nodes)} nodes and {len(subgraph.edges)} edges.\n")
        f.write("Markdown tree representation:\n")
        f.write(graph_to_markdown_tree(subgraph))
    print(f"Subgraph statistics saved to {file_name}")

In [43]:
def find_function_source_in_file(path: str, function_name: str) -> str | None:
    sig_re = re.compile(rf'\b{re.escape(function_name)}\b\s*\(')
    def_re = re.compile(
        rf'''^[ \t]*
            (?:[A-Za-z_]\w*(?:\s*\*+)?\s+)+
            \**\s*
            {re.escape(function_name)}\s*\(
        ''',
        re.VERBOSE
    )
    try:
        f = open(path, 'r', errors='ignore')
    except PermissionError:
        return None
    with f:
        collecting = False
        brace_count = 0
        buffer = []
        in_block = False
        for raw in f:
            line = raw
            if in_block:
                end = line.find('*/')
                if end >= 0:
                    in_block = False
                    line = line[end+2:]
                else:
                    continue
            start = line.find('/*')
            if start >= 0:
                end = line.find('*/', start+2)
                if end >= 0:
                    line = line[:start] + line[end+2:]
                else:
                    in_block = True
                    line = line[:start]
            clean = line.split('//',1)[0]
            if clean.rstrip().endswith('\\'):
                continue
            if not collecting:
                if sig_re.search(clean) and def_re.match(clean):
                    sig_raw = [raw]
                    sig_clean = [clean]
                    bal = clean.count('(') - clean.count(')')
                    while bal > 0:
                        nxt_raw = f.readline()
                        if not nxt_raw:
                            break
                        nxt = nxt_raw
                        if in_block:
                            end = nxt.find('*/')
                            if end >= 0:
                                in_block = False
                                nxt = nxt[end+2:]
                            else:
                                continue
                        st = nxt.find('/*')
                        if st >= 0:
                            ed = nxt.find('*/', st+2)
                            if ed >= 0:
                                nxt = nxt[:st] + nxt[ed+2:]
                            else:
                                in_block = True
                                nxt = nxt[:st]
                        nxt_clean = nxt.split('//',1)[0]
                        sig_raw.append(nxt_raw)
                        sig_clean.append(nxt_clean)
                        bal += nxt_clean.count('(') - nxt_clean.count(')')
                    if ''.join(sig_clean).strip().endswith(';'):
                        continue
                    buffer = sig_raw.copy()
                    if '{' in sig_clean[-1]:
                        collecting = True
                        brace_count = sig_clean[-1].count('{') - sig_clean[-1].count('}')
                    else:
                        for body_raw in f:
                            body = body_raw
                            if in_block:
                                end = body.find('*/')
                                if end >= 0:
                                    in_block = False
                                    body = body[end+2:]
                                else:
                                    continue
                            st = body.find('/*')
                            if st >= 0:
                                ed = body.find('*/', st+2)
                                if ed >= 0:
                                    body = body[:st] + body[ed+2:]
                                else:
                                    in_block = True
                                    body = body[:st]
                            body_clean = body.split('//',1)[0]
                            buffer.append(body_raw)
                            if '{' in body_clean:
                                collecting = True
                                brace_count = (
                                    body_clean.count('{') -
                                    body_clean.count('}')
                                )
                                break
                    if not collecting:
                        buffer = []
            else:
                buffer.append(raw)
                tmp = raw
                if in_block:
                    end = tmp.find('*/')
                    if end >= 0:
                        in_block = False
                        tmp = tmp[end+2:]
                    else:
                        continue
                st = tmp.find('/*')
                if st >= 0:
                    ed = tmp.find('*/', st+2)
                    if ed >= 0:
                        tmp = tmp[:st] + tmp[ed+2:]
                    else:
                        in_block = True
                        tmp = tmp[:st]
                tmp_clean = tmp.split('//',1)[0]
                brace_count += tmp_clean.count('{') - tmp_clean.count('}')
                if brace_count == 0:
                    return ''.join(buffer)
    return None

def find_function_native(root_dir: str, function_name: str) -> str | None:
    pat = rf'{re.escape(function_name)}[[:space:]]*\('
    try:
        out = subprocess.check_output([
            'grep', '-RlE',
            '--include=*.c', pat, root_dir
        ], text=True, stderr=subprocess.DEVNULL)
    except subprocess.CalledProcessError:
        return None
    for file_path in out.splitlines():
        src = find_function_source_in_file(file_path, function_name)
        if src:
            return src
    return None

In [44]:
# profile file is always nginx-ltp-redis
profiled_functions = extract_profiled_functions(os.path.join(s_src_dir, "nginx-ltp-redis"))
print(f"Extracted {len(profiled_functions)} profiled functions from {s_src_dir}.")

Extracted 426 profiled functions from syscall_profiles/3:close.


### Extract all kernel functions' source code

This is a one-time effort by scanning the `linux/` code directory.

In [ ]:
from tqdm import tqdm
from multiprocessing import Pool, cpu_count

# Generate the_functions_all.txt for the currently processing syscall

graph_for_proc = k_cg_acyc.copy()
print(f"Kernel-CFG (acyclic) with {len(graph_for_proc.nodes())} nodes and {len(graph_for_proc.edges())} edges.")
the_dictionary_functions = {}

ending = 'XXXTHISENDSHEREXXX'

def process_node(node):
    src = find_function_native(linux_src_directory, node)
    if src is not None:
        return node, src
    return None

with open(func_src_file, "w") as f:
    f.write("")

nodes = list(graph_for_proc.nodes())
with Pool(cpu_count()) as pool:
    results = list(tqdm(pool.imap(process_node, nodes), total=len(nodes), desc="Processing nodes"))

for result in results:
    if result is not None:
        node, src = result
        the_dictionary_functions[node] = src
        with open(func_src_file, "a") as f:
            f.write(f"Source Code for {node}:\n")
            f.write(src)
            f.write(f"{ending}\n\n")
            
print('Done!')

Kernel-CFG (acyclic) with 41879 nodes and 237126 edges.


Processing nodes: 100%|██████████| 41879/41879 [04:46<00:00, 146.15it/s]


Done!


In [ ]:
# we first dump it's pure static call subgraph
pure_static_subgraph = gen_subgraph(k_cg_acyc, sys_entry_function=f"__x64_sys_{syscall}", function_set=None)
print(f"Pure static subgraph for {syscall} has {len(pure_static_subgraph.nodes)} nodes and {len(pure_static_subgraph.edges)} edges.")

dump_subgraph_statistics(pure_static_subgraph, os.path.join(s_proc_dir, proc_pure_static_file))

prof_static_subgraph = gen_subgraph(k_cg_acyc, sys_entry_function=f"__x64_sys_{syscall}", function_set=profiled_functions)
print(f"Profiled static subgraph for {syscall} has {len(prof_static_subgraph.nodes)} nodes and {len(prof_static_subgraph.edges)} edges.")

dump_subgraph_statistics(prof_static_subgraph, os.path.join(s_proc_dir, proc_profile_static_file))

Pure static subgraph for close has 2205 nodes and 4871 edges.
Subgraph statistics saved to syscall_procs/3:close/pure-static.txt
Profiled static subgraph for close has 3704 nodes and 26472 edges.
Subgraph statistics saved to syscall_procs/3:close/profile-static.txt


In [51]:
# Load kernel's source code blocks
content = None
with open("the_functions_all.txt", "r") as f:
    content = f.read()

ending = 'XXXTHISENDSHEREXXX'
pattern = rf"Source Code for\s+([\w_]+)\s*:\s*\n(.*?)(?={re.escape(ending)})"
all_blocks = dict(re.findall(pattern, content, flags=re.DOTALL))
print(f"Loaded {len(all_blocks)} function source blocks from the_functions_all.txt.")

Loaded 37200 function source blocks from the_functions_all.txt.


### Last step: the algorithm

This is the function to query LLM.

In [55]:
from tqdm import tqdm

def LLM_hybrid_expand_profile(
    subgraph: nx.DiGraph,
    k_cg: nx.DiGraph,
    all_blocks: dict[str, str],
    N: int=2,
    model: str="qwen3:32b",
    role: str="user",
    num_ctx: int=14336,
    syscall_info: str="3:close") -> nx.DiGraph:
    """
    Proof-of-concept: iteratively expand `subgraph` by querying LLM for
    semantically valid static edges, up to N hops per dynamic start node.
    """
    
    out_dir = f"{proc_direcotry}/{syscall_info}"
    # log output
    the_output = f"{out_dir}/llm_logs.txt"
    
    open(the_output, 'w', encoding='utf-8').close()
    queried = set()

    # Topo sort dynamic subgraph and process bottom-up
    dynamic_order = list(nx.topological_sort(subgraph))[::-1]

    # Initialize progress bar
    with tqdm(total=len(dynamic_order) * N, desc="Expanding subgraph") as pbar:
        for D in dynamic_order:
            # Initialize frontier at depth 0
            frontier = [D]
            for depth in range(N):
                if not frontier:
                    break
                next_frontier = []
                markdown_output = graph_to_markdown_tree(subgraph)

                # Pre-gather sources for current subgraph
                function_sources = {node: all_blocks.get(node) for node in subgraph.nodes}

                for src in frontier:
                    # Skip if the source node has no outgoing edges in the full call graph
                    if not k_cg.has_node(src):
                        continue

                    # Static callees from full call graph
                    for _, dst in k_cg.out_edges(src):
                        if subgraph.has_edge(src, dst) or (src, dst) in queried:
                            continue
                        queried.add((src, dst))

                        # Build prompt
                        prompt_text = (
                            f"You are a Linux security expert analyzing kernel call-graph edges.\n"
                            f"Historical dynamic call graph:\n{markdown_output}\n\n"
                            f"Caller: {src}\nSource code:\n{function_sources.get(src)}\n\n"
                            f"Callee candidate: {dst}\nSource code:\n{all_blocks.get(dst)}\n\n"
                            f"Additional context (other functions):\n"
                        )
                        for node, code in function_sources.items():
                            prompt_text += f"-- {node}:\n{code}\n"
                        prompt_text += (
                            f"\nFrom a security-engineering standpoint, is it reasonable to expect that "
                            f"execution of {src} will reach {dst}? Provide a concise justification, "
                            f"then a literal answer: '{{Your justification}}\nFINAL ANSWER -> YES/NO'"
                        )

                        # Log prompt
                        with open(the_output, 'a', encoding='utf-8') as f:
                            f.write(f"PROMPT (depth {depth}):\n{prompt_text}\n\n")

                        # Query LLM
                        response = chat(
                            model=model,
                            messages=[{'role': role, 'content': prompt_text}],
                            options={'num_ctx': num_ctx}
                        )
                        resp = re.sub(r'<think>.*?</think>', '', response.message.content, flags=re.DOTALL).strip()
                        resp = re.sub(r'[ \t]+$', '', resp, flags=re.MULTILINE)

                        # Log response
                        with open(the_output, 'a', encoding='utf-8') as f:
                            f.write(f"RESPONSE:\n{resp}\n\n")

                        if 'FINAL ANSWER -> YES' in resp.upper():
                            subgraph.add_node(dst)
                            subgraph.add_edge(src, dst, inferred=True)
                            next_frontier.append(dst)
                            with open(the_output, 'a', encoding='utf-8') as f:
                                f.write(f"Added edge: {src} -> {dst}\n")

                frontier = next_frontier
                pbar.update(1)  # Update progress bar

    # Dump statistics of final subgraph
    # final_md = graph_to_markdown_tree(subgraph)
    # with open(f"{the_directory}/the_real_markdown.txt", 'w', encoding='utf-8') as f:
    #     f.write(final_md)
    dump_subgraph_statistics(subgraph, os.path.join(out_dir, "result.txt"))
    return subgraph

In [ ]:
dyn_graph = gen_subgraph(k_cg_acyc, sys_entry_function=None, function_set=profiled_functions)
LLM_hybrid_expand_profile(
    subgraph=dyn_graph,
    k_cg=k_cg_acyc,
    all_blocks=all_blocks,
    N=2,
    model="qwen3:32b",
    role="user",
    num_ctx=14336,
    syscall_info=f"{sys_id}:{syscall}"
)

TypeError: LLM_hybrid_expand_profile() missing 3 required positional arguments: 'subgraph', 'k_cg', and 'all_blocks'